# Agentic AI notebook

Ez a notebook egy minimalis WebShop Pro support agentet epit tool calling gondolkodassal. A teljes lab profil:

```bash
docker compose --profile agentic up -d
```

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Tool:
    name: str
    description: str
    fn: Callable[..., str]

ORDERS = {
    'O-10031': 'paid, futarnal, varhato erkezes: holnap',
    'O-10034': 'refunded, visszautalas folyamatban',
}

def get_order_status(order_id: str) -> str:
    return ORDERS.get(order_id, 'nincs ilyen rendeles a demo adatban')

def check_return_policy(order_id: str) -> str:
    return '14 napos elallas ervenyes, ha a termek serulesmentes es nem egyedi konfiguracio'

tools = {
    'get_order_status': Tool('get_order_status', 'Rendelesi statusz lekerdezese', get_order_status),
    'check_return_policy': Tool('check_return_policy', 'Visszakuldesi policy ellenorzese', check_return_policy),
}
tools

In [ ]:
def plan(message: str):
    steps = []
    order_id = next((part.strip(',.!?') for part in message.split() if part.startswith('O-')), None)
    if order_id:
        steps.append(('get_order_status', {'order_id': order_id}))
        if 'vissza' in message.lower() or 'return' in message.lower():
            steps.append(('check_return_policy', {'order_id': order_id}))
    return steps

plan('Hol van az O-10031 rendelesem, es vissza tudom kuldeni?')

In [ ]:
def run_agent(message: str):
    trace = []
    observations = []
    for tool_name, kwargs in plan(message):
        tool = tools[tool_name]
        result = tool.fn(**kwargs)
        trace.append({'tool': tool_name, 'args': kwargs, 'observation': result})
        observations.append(result)
    answer = ' | '.join(observations) if observations else 'Kerlek add meg a rendelesszamot is.'
    return {'answer': answer, 'trace': trace}

run_agent('Hol van az O-10031 rendelesem, es vissza tudom kuldeni?')

In [ ]:
def evaluate_trace(result):
    tool_names = [step['tool'] for step in result['trace']]
    return {
        'uses_order_tool': 'get_order_status' in tool_names,
        'uses_policy_tool_when_needed': 'check_return_policy' in tool_names,
        'has_final_answer': bool(result['answer']),
    }

result = run_agent('Hol van az O-10031 rendelesem, es vissza tudom kuldeni?')
evaluate_trace(result)

Production irany: ugyanezt LangGraph state graphban, FastAPI endpoint mogott es LangSmith/Langfuse trace-ekkel erdemes tovabbvinni.